# EDA on marathos dataset

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(f"{VOLUME_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv")
)

In [0]:
from pyspark.sql import functions as F

display(df.limit(10))
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

###### Count rows and columns

In [0]:
num_rows = df.count()
num_cols = len(df.columns)

print(f"Antal rader    : {num_rows:,}")
print(f"Antal kolumner : {num_cols}")

###### Schema

In [0]:
print("schema: ")
df.printSchema()

###### Numerical columns

In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, NumericType)
]

print(f"\nNumeriska kolumner ({len(numeric_cols)}):")
print(numeric_cols)

df.select(numeric_cols).describe().show(truncate=False)

###### Count nulls

In [0]:
from pyspark.sql.functions import col, sum as _sum, isnan, when #best practice att byta namn på sum 

nulls = df.select([
    _sum(
        when(col(c).isNull() | (col(c).cast("string") == ""), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

print("Nulls per kolumn:")
nulls.show(vertical=True, truncate=False)

###### Unique events

In [0]:
# from pyspark.sql.functions import desc
 
# unique_events = df.select("Event name").distinct().count()
# print(f"Antal unika event: {unique_events:,}")
 
# # Top 20 mest förekommande event
# print("\nTop 20 vanligaste event:")
# (
#     df.groupBy(EVENT_COL)
#     .count()
#     .orderBy(desc("count"))
#     .limit(20)
#     .show(truncate=False)
# )

###### Age distribution - Runners

In [0]:
from pyspark.sql.functions import floor, repeat, lit


df_age = (
    df
    # Ta bort rader där födelseår saknas
    .filter(col("Athlete year of birth").isNotNull())
    # Skapa ny kolumn "age" genom att subtrahera födelseåret från nuvarande år
    .withColumn("age", col("Year of event") - col("Athlete year of birth").cast("integer"))
    # Filtrera bort orimliga värden – negativa åldrar eller 0
    .filter(col("age") > 0)
    # Filtrera bort orimliga värden – ingen människa är 120+ år
    .filter(col("age") < 120)
)

# Visa grundläggande statistik: count, mean, stddev, min, max för ålder
print("Åldersstatistik:")
df_age.select("age").describe().show()



###### Landspresentation

In [0]:
print("Top 10 länder efter antal starter:")
(
    df.filter(col("Athlete country").isNotNull())
    .groupBy("Athlete country")
    .count()
    .orderBy(desc("count"))
    .limit(10)
    .show(truncate=False)
)
 
# Totalt antal unika länder
unique_countries = df.select("Athlete country").distinct().count()
print(f"Totalt antal unika länder: {unique_countries}")

###### Gender

In [0]:
from pyspark.sql.functions import round as spark_round

# Räkna antal kvinnor och män per race i en enda groupBy
gender_pivot = (
    df
    # Ta bort rader där kön eller race saknas
    .filter(col("Athlete gender").isNotNull())
    .filter(col("Event name").isNotNull())
    .groupBy("Event name")
    .agg(
        # Räkna bara rader där kön är W
        _sum(when(col("Athlete gender") == "F", 1).otherwise(0)).alias("women"),
        # Räkna bara rader där kön är M
        _sum(when(col("Athlete gender") == "M", 1).otherwise(0)).alias("men"),
        # Räkna totalt antal starter per race
        _sum(lit(1)).alias("total")
    )
    # Beräkna procentandel kvinnor, avrundat till 1 decimal
    .withColumn("procent_kvinna", spark_round((col("women") / col("total")) * 100, 1))
    # Beräkna procentandel män, avrundat till 1 decimal
    .withColumn("procent_man", spark_round((col("men") / col("total")) * 100, 1))
    # Behåll bara de kolumner vi vill visa
    .select("Event name", "procent_kvinna", "procent_man", "total")
    # Sortera på störst race först
    .orderBy(desc("total"))
)

gender_pivot.show(10, truncate=False)

In [0]:
df.select("Athlete country").distinct().display()